In [0]:
import os
import sys
import uuid
import json
import logging
import time
import threading
import asyncio
import structlog
import uvicorn
from fastapi import FastAPI, status, HTTPException
from pydantic import BaseModel, Field

# — NEW: AZURE MONITOR PIPELINE CONFIGURATION —
import contextlib
import io
from azure.monitor.opentelemetry import configure_azure_monitor

# Paste your copied string here (or use databricks secrets utility)
AZURE_CONN_STR = "InstrumentationKey=cd9c0655-9f68-4f18-99ee-cc639425e3fe;IngestionEndpoint=https://centralindia-0.in.applicationinsights.azure.com/;LiveEndpoint=https://centralindia.livediagnostics.monitor.azure.com/;ApplicationId=699469eb-7f1c-4fb6-8236-4003bad64881"

# Guard against double-configuration on cell re-run; suppress non-fatal Azure SDK tracing warning
if not globals().get("_azure_monitor_configured"):
    with contextlib.redirect_stdout(io.StringIO()):
        configure_azure_monitor(
            connection_string=AZURE_CONN_STR,
            logger_name="fastapi_routing_app"  # Isolates app metrics from underlying SDK trace logs
        )
    _azure_monitor_configured = True
# —

# 1. Structured Logging Configuration
import logging
import sys

logging.basicConfig(format="%(message)s", stream=sys.stdout, level=logging.INFO)
structlog.reset_defaults()
structlog.configure(
    processors=[
        structlog.stdlib.add_log_level,
        structlog.processors.TimeStamper(fmt="iso"),
        structlog.processors.JSONRenderer()  # Log Analytics can naturally parse this JSON string
    ],
    context_class=dict,
    logger_factory=structlog.stdlib.LoggerFactory(),
    wrapper_class=structlog.stdlib.BoundLogger,
    cache_logger_on_first_use=True,
)
logger = structlog.get_logger("fastapi_routing_app")

# 2. FastAPI Application Setup
app = FastAPI(title="Connected Azure Log Analytics App")

class TargetRequest(BaseModel):
    user_name: str
    payload: dict

def run_ml_inference(data: dict) -> dict:
    logger.info("Starting ML model execution...", approach="neural_network_inference")
    time.sleep(0.150)
    return {"predicted_class": 0, "confidence": 0.94}

async def run_simple_pipeline(data: dict) -> dict:
    logger.info("Executing standard database processing pipeline...")
    return {"status": "processed"}

@app.post("/execute", status_code=status.HTTP_200_OK)
async def route_user_request(request: TargetRequest):
    user_name = request.user_name
    # Explicit logging context tracking
    log = logger.bind(user_name=user_name, request_id=str(uuid.uuid4()))

    if "ml_user" in user_name:
        log.info("Routing request to ML inference pool")
        result = await asyncio.to_thread(run_ml_inference, request.payload)
        log.info("ML inference successfully completed")
        return {"routing": "ml_pipeline", "result": result}
    elif "db_user" in user_name:
        log.info("Routing request to standard database function")
        result = await run_simple_pipeline(request.payload)
        log.info("Database execution successfully completed")
        return {"routing": "database_pipeline", "result": result}
    else:
        raise HTTPException(status_code=400, detail="Invalid user naming convention.")